# ШАД, Рекомендательные системы 2026

## Бонусное домашнее задание по теме Semantic IDs

- Матчасть (база): https://arxiv.org/pdf/2305.05065
- Подходы с Semantic IDs можно рассматривать как альтернативу классической постановке next-item prediction через классификацию по всему каталогу item-ов. Вместо того чтобы предсказывать один item_id из большого словаря, модель предсказывает короткую последовательность дискретных токенов:

$$
s_i = (s_{i,1}, s_{i,2}, \dots, s_{i,M}).
$$

- Каждый item получает свой Semantic ID, а задача рекомендации превращается в задачу генерации последовательности токенов. Это позволяет избежать вычисления полного softmax над большим каталогом товаров, а также повысить обобщающую способность модели.
- В данной домашней работе мы реализуем 2 самых популярных способа построения Semantic IDs, а также обучим трансформеры, использующие эти IDs.

### Оценивание

- Задание 1. RQ-KMeans — 2 балла
- Задание 2. Разрешение коллизий — 1 балл
- Задание 3. Semantic Transformer — 2 балла
- Задание 4. Beam search — 1 балл
- Задание 5. Обучение модели с коллизиями и без них — 1 балл
- Задание 6. RQ-VAE — 2 балла
- Задание 7. Сравнение всех моделей — 1 балл

**Итого: 10 баллов.**

Дополнительные баллы:
- +2 балла за качество выше full softmax baseline.
- +2 балла за качество RQ-VAE Semantic IDs выше RQ-KMeans Semantic IDs.

## Данные для работы:

- Работать будем с лайками датасета Yambda.
- Для ускорения обучения мы берём 5 млн последних по времени взаимодействий.

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

import polars as pl
import tests

In [ ]:
file_path = "likes.parquet"

count_of_samples = 5000000

data = kagglehub.dataset_load(
  KaggleDatasetAdapter.POLARS,
  "thekabeton/ysda-recsys-2026-yambda-dataset",
  file_path
).collect().sort('timestamp', descending=True)[:count_of_samples]

data

- Чаще всего, для построения Semantic IDs используются контентные эмбеддинги.
- Нам повезло, что такие как раз есть в нашем датасете. Будем использовать их.

In [ ]:
from datasets import load_dataset

file_path = "embeddings_small.parquet"

embeddings = kagglehub.dataset_load(
  KaggleDatasetAdapter.POLARS,
  "thekabeton/ysda-recsys-2026-yambda-dataset",
  file_path
).collect()

embeddings

In [ ]:
item_ids = data.select('item_id').unique().get_column('item_id')

embeddings = embeddings.filter(
    pl.col('item_id').is_in(item_ids.implode())
)['item_id', 'normalized_embed']

embeddings

- Оставляем только те item-ы, для которых есть контентные эмбеддинги.

In [ ]:
items_with_embeddings = embeddings["item_id"].unique()

data = data.filter(
    pl.col("item_id").is_in(items_with_embeddings.implode())
)

data

### Задание 1: (2 балла)

- Самым простым способом построения Semantic IDs из эмбеддингов является **RQ-KMeans**. Этот подход отличается от оригинальной идеи, которая была предложена в статье [TIGER](https://arxiv.org/pdf/2305.05065) от DeepMind, но является более простой для реализации. Данный алгоритм долгое время встречался в статьях от китайской компании [Kuaishou](https://arxiv.org/html/2506.13695v1).

- Идея RQ-KMeans заключается в том, чтобы кластеризовывать эмбеддинги айтемов с помощью [KMeans](https://en.wikipedia.org/wiki/K-means_clustering) и использовать ближайшие кластеры в качествe Semantic IDs. После получения кластеров, из каждого эмбеддинга вычитается эмбеддинг его кластера и для полученных остатков (residuals) алгоритм повторяется.

-  Первый кластер описывает грубую семантику объекта, второй кластер уточняет ошибку первого приближения, третий уточняет ошибку второго и так далее. Поэтому полученный Semantic ID имеет иерархическую структуру: первые токены отвечают за coarse-level semantics, а последние — за более fine-grained различия между айтемами.

---

### Алгоритм RQ-KMeans формально:

Пусть у нас есть матрица item embeddings:

$$
X = \{x_i\}_{i=1}^{N}, \quad x_i \in \mathbb{R}^{d},
$$

где $N$ — количество айтемов, а $d$ — размерность эмбеддинга.

Мы хотим построить для каждого айтема дискретный Semantic ID длины $M$:

$$
s_i = (s_{i,1}, s_{i,2}, \dots, s_{i,M}),
$$

где каждый $s_{i,m}$ — номер кластера на уровне $m$:

$$
s_{i,m} \in \{0, 1, \dots, K_m - 1\}.
$$

На первом уровне мы запускаем обычный KMeans по исходным эмбеддингам:

$$
s_{i,1} = \arg\min_{k} \|x_i - c_{1,k}\|_2^2,
$$

где $c_{1,k}$ — центроид $k$-го кластера в первом codebook.

После этого мы аппроксимируем эмбеддинг первым центроидом:

$$
\hat{x}_i^{(1)} = c_{1, s_{i,1}}.
$$

Затем считаем residual, то есть ошибку восстановления:

$$
r_i^{(1)} = x_i - \hat{x}_i^{(1)}.
$$

На втором уровне мы запускаем KMeans уже не по исходным эмбеддингам, а по residual-векторам:

$$
s_{i,2} = \arg\min_{k} \|r_i^{(1)} - c_{2,k}\|_2^2.
$$

После этого обновляем восстановленный эмбеддинг:

$$
\hat{x}_i^{(2)} = c_{1, s_{i,1}} + c_{2, s_{i,2}},
$$

и снова считаем residual:

$$
r_i^{(2)} = x_i - \hat{x}_i^{(2)}.
$$

В общем виде для уровня $m$:

$$
s_{i,m} = \arg\min_{k} \|r_i^{(m-1)} - c_{m,k}\|_2^2,
$$

где

$$
r_i^{(m-1)} = x_i - \sum_{j=1}^{m-1} c_{j, s_{i,j}}.
$$

После $M$ уровней каждый айтем получает Semantic ID:

$$
s_i = (s_{i,1}, s_{i,2}, \dots, s_{i,M}).
$$

---

### Что нужно реализовать

В этом задании вам нужно реализовать класс RQ-KMeans для построения Semantic IDs из item embeddings.

- При обучении класс получает item embeddings

- На выходе мы хотим получать Polars DF с колонками: **item_id, sid_0, sid_1, sid_2**

- В `get_semantic_ids` лучше не строить матрицу расстояний для всех item-ов сразу: она может занимать много памяти. Обрабатывайте embeddings батчами.

In [ ]:
import numpy as np
import polars as pl

from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import pairwise_distances_argmin


class RQKMeans:
    def __init__(
        self,
        num_levels=3,
        codebook_size=512,
        random_state=42,
        kmeans_batch_size=1024 * 16,
        predict_batch_size=65536,
    ):
        self.num_levels = num_levels
        self.codebook_size = codebook_size
        self.random_state = random_state
        self.kmeans_batch_size = kmeans_batch_size
        self.predict_batch_size = predict_batch_size
        self.codebooks = []

    def fit(self, X):
        X = ...
        residual = ...
        self.codebooks = []

        for level in range(self.num_levels):
            # TODO: train k-means on residual
            ...

            # TODO: update residual and save codebook
            ...

        return self

    def get_semantic_ids(self, X):
        X = ...
        codes_all = ...

        for start in range(...):
            # TODO: process one batch
            ...

            for level, centers in enumerate(self.codebooks):
                # TODO: assign nearest centers and update residual
                ...

        return codes_all

    def fit_transform_items(self, items_df):
        item_ids = ...
        X = ...

        # TODO: fit model and get codes
        ...

        return pl.DataFrame({
            "item_id": item_ids,
            # TODO: add sid_0, sid_1, ...
        })

In [ ]:
rq = RQKMeans(num_levels=3, codebook_size=512)
semantic_df = rq.fit_transform_items(embeddings)

print(semantic_df.head())

### Проверка

- Должны проходить все тесты
- unique semantic ids должен быть больше 300к

In [ ]:
tests.check_semantic_ids(semantic_df)

## Коллизии

- Во время построения Semantic IDs могут возникать коллизии: несколько item-ов получают одинаковую последовательность semantic-токенов.

Например:

$$
s_a = (12, 45, 7), \quad s_b = (12, 45, 7).
$$

Один простой способ разрешить такие коллизии — добавить дополнительный токен в конец Semantic ID. Этот токен не является семантическим: он просто нумерует item-ы внутри одной коллизии.

Например:

$$
(12, 45, 7) \rightarrow (12, 45, 7, 0)
$$

и

$$
(12, 45, 7) \rightarrow (12, 45, 7, 1).
$$

Важно: такой токен может ухудшать качество generative-модели, потому что один и тот же токен могут получать совершенно разные айтемы, полученные случайным образом.

### Задание 2: разрешение коллизий (1 балл)

Добавьте в `semantic_df` дополнительный токен `sid_3`, который делает полный Semantic ID уникальным для каждого item-а.

In [ ]:
def get_sid_cols(semantic_df):
    return sorted(
        [c for c in semantic_df.columns if c.startswith("sid_")],
        key=lambda x: int(x.split("_")[1]),
    )

def add_collision_resolving_sid(
    semantic_df: pl.DataFrame,
    item_col: str = "item_id",
    new_col: str | None = None,
):
    sid_cols = get_sid_cols(semantic_df)

    if new_col is None:
        new_col = ...

    # TODO: assign local index inside each group with same SID prefix
    df = ...

    return df

semantic_df_resolved = add_collision_resolving_sid(
    semantic_df,
    item_col="item_id",
)

semantic_df_resolved.head()

### Тесты:

In [ ]:
tests.check_collision_resolving_sid(
    old_df=semantic_df,
    new_df=semantic_df_resolved,
)

## Бейзлайны

- Мы предоставляем две базовые модели:
    1. **Full softmax model** — модель предсказывает следующий item через softmax по всему каталогу.
    2. **Sampled softmax model** — модель считает loss только на положительном item-е и наборе sampled negatives.

--- 

Пусть encoder истории возвращает user-вектор:

$$
u = f_\theta(h_1, h_2, \dots, h_T).
$$

В full softmax модели score для item-а $j$ равен:

$$
z_j = u^\top e_j,
$$

где $e_j$ — embedding item-а. Loss считается по всем item-ам:

$$
\mathcal{L}_{full} = -\log \frac{\exp(z_y)}{\sum_{j \in \mathcal{I}} \exp(z_j)}.
$$

В sampled softmax denominator считается только по подмножеству кандидатов:

$$
\mathcal{C} = \{y\} \cup \mathcal{N},
$$

где $\mathcal{N}$ — sampled negatives.

---

- Эти модели нужны как baseline для сравнения с моделью, которая генерирует Semantic IDs.
- Вам необходимо обучить эти модели, чтобы получить метрики для сравнения

In [ ]:
import random
from dataclasses import dataclass

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Конфиг обучения моделей, используйте заданные параметры
@dataclass
class Config:
    max_history_len: int = 16
    min_history_len: int = 5
    seconds_per_day: int = 86_400
    val_num_days: int = 1

    d_model: int = 128
    d_ff: int = 512
    num_heads: int = 4
    num_layers: int = 4
    dropout: float = 0.1

    batch_size: int = 1024
    batch_size_val: int = 64
    lr: float = 1e-3
    weight_decay: float = 1e-2
    num_epochs: int = 1
    grad_clip: float = 1.0

    topk: int = 10

    num_levels: int = 3
    codebook_size: int = 512
    num_beams: int = 100

    log_every_n_steps: int = 100
    val_subset_size: int | None = 50_000

    num_negatives: int = 2048

    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

### Обработчики датасета

In [ ]:
def prepare_examples(data, cfg):
    item_ids = sorted(data['item_id'].unique().to_list())
    item2idx = {item_id: i + 1 for i, item_id in enumerate(item_ids)}
    idx2item = {idx: item_id for item_id, idx in item2idx.items()}

    df = (
        data.with_columns((pl.col("timestamp") // cfg.seconds_per_day).alias("day"))
        .sort(["uid", "timestamp", "item_id"])
    )

    days = sorted(df["day"].unique().to_list())
    val_days = set(days[-cfg.val_num_days:])

    examples = []
    history = []
    current_uid = None

    for row in df.iter_rows(named=True):
        uid = int(row["uid"])
        item_id = int(row["item_id"])
        day = int(row["day"])

        if uid != current_uid:
            current_uid = uid
            history = []

        if item_id not in item2idx:
            continue

        item_idx = item2idx[item_id]

        if len(history) >= cfg.min_history_len:
            examples.append({
                "history": history[-cfg.max_history_len:],
                "target": item_idx,
                "is_val": day in val_days,
            })

        history.append(item_idx)

    train = [x for x in examples if not x["is_val"]]
    val = [x for x in examples if x["is_val"]]

    return train, val, item2idx, idx2item


class RecDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def collate_fn(batch):
    bsz = len(batch)
    max_len = max(len(x["history"]) for x in batch)

    history = torch.zeros(bsz, max_len, dtype=torch.long)
    positions = torch.zeros(bsz, max_len, dtype=torch.long)
    mask = torch.zeros(bsz, max_len, dtype=torch.bool)

    target = torch.tensor([x["target"] for x in batch], dtype=torch.long)
    for i, x in enumerate(batch):
        h = torch.tensor(x["history"], dtype=torch.long)
        L = len(h)

        history[i, :L] = h
        positions[i, :L] = torch.arange(L)
        mask[i, :L] = True

    return {
        "history": history,
        "positions": positions,
        "mask": mask,
        "target": target,
    }


def limit_examples(examples, n, seed=42):
    if n is None or len(examples) <= n:
        return examples

    rng = random.Random(seed)
    idx = rng.sample(range(len(examples)), n)
    return [examples[i] for i in sorted(idx)]


def to_device(batch, device):
    return {k: v.to(device) for k, v in batch.items()}

### Энкодер общий для всех моделей

In [ ]:
class HistoryTransformerEncoder(nn.Module):
    def __init__(self, num_items, cfg):
        super().__init__()

        self.item_emb = nn.Embedding(num_items + 1, cfg.d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(cfg.max_history_len, cfg.d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.num_heads,
            dim_feedforward=cfg.d_ff,
            dropout=cfg.dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu",
        )

        self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.num_layers)
        self.norm = nn.LayerNorm(cfg.d_model)

    def forward(self, history, positions, mask):
        x = self.item_emb(history) + self.pos_emb(positions)

        x = self.encoder(
            x,
            src_key_padding_mask=~mask,
        )

        x = self.norm(x)

        pooled = (x * mask.unsqueeze(-1)).sum(1)
        pooled = pooled / mask.sum(1, keepdim=True).clamp_min(1)

        return x, pooled

### Модель на полном Softmax

In [ ]:
class FullSoftmaxModel(nn.Module):
    def __init__(self, num_items, cfg):
        super().__init__()
        self.encoder = HistoryTransformerEncoder(num_items, cfg)
        self.head = nn.Linear(cfg.d_model, num_items + 1)

    def forward(self, history, positions, mask):
        _, pooled = self.encoder(history, positions, mask)
        logits = self.head(pooled)
        return logits

### Модель с семплированными негативами

In [ ]:
class SampledSoftmaxModel(nn.Module):
    def __init__(self, num_items, cfg):
        super().__init__()
        self.num_items = num_items
        self.encoder = HistoryTransformerEncoder(num_items, cfg)

    def forward(self, history, positions, mask, target, num_negatives):
        _, pooled = self.encoder(history, positions, mask)

        negatives = torch.randint(
            1,
            self.num_items + 1,
            size=(target.size(0), num_negatives),
            device=target.device,
        )

        candidates = torch.cat([target[:, None], negatives], dim=1)
        candidate_embs = self.encoder.item_emb(candidates)

        logits = (pooled[:, None, :] * candidate_embs).sum(-1)
        labels = torch.zeros(target.size(0), dtype=torch.long, device=target.device)

        return logits, labels

    @torch.no_grad()
    def full_logits(self, history, positions, mask):
        _, pooled = self.encoder(history, positions, mask)
        logits = pooled @ self.encoder.item_emb.weight.T
        return logits

### Валидация модели на айдишниках

In [ ]:
@torch.no_grad()
def validate_item_model(model, val_loader, cfg, sampled=False):
    model.eval()

    total = 0
    hits = 0

    for batch in val_loader:
        batch = to_device(batch, cfg.device)

        if sampled:
            logits = model.full_logits(
                batch["history"],
                batch["positions"],
                batch["mask"],
            )
        else:
            logits = model(
                batch["history"],
                batch["positions"],
                batch["mask"],
            )

        logits[:, 0] = -1e9

        topk = logits.topk(cfg.topk, dim=1).indices
        hit = (topk == batch["target"][:, None]).any(1)

        total += batch["target"].size(0)
        hits += hit.sum().item()

    return {
        f"recall@{cfg.topk}": hits / max(total, 1),
    }

### Трейн лупы

In [ ]:
def train_full_softmax(model, train_loader, val_loader, cfg):
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )

    history = []

    for epoch in range(cfg.num_epochs):
        model.train()

        for step, batch in enumerate(train_loader, start=1):
            batch = to_device(batch, cfg.device)

            logits = model(
                batch["history"],
                batch["positions"],
                batch["mask"],
            )

            loss = F.cross_entropy(logits, batch["target"])

            opt.zero_grad()
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                cfg.grad_clip,
            )
            opt.step()

            if step % cfg.log_every_n_steps == 0:
                print(
                    f"full softmax | "
                    f"epoch={epoch + 1}/{cfg.num_epochs} | "
                    f"step={step}/{len(train_loader)} | "
                    f"loss={loss.item():.6f} | "
                    f"grad_norm={float(grad_norm):.6f}"
                )

        metrics = validate_item_model(
            model,
            val_loader,
            cfg,
            sampled=False,
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": float(loss.item()),
            **metrics,
        })

        print(
            f"full softmax | "
            f"epoch={epoch + 1} done | "
            f"loss={loss.item():.6f} | "
            f"recall@{cfg.topk}={metrics[f'recall@{cfg.topk}']:.6f}"
        )

    return history

In [ ]:
def train_sampled_softmax(model, train_loader, val_loader, cfg):
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )

    history = []

    for epoch in range(cfg.num_epochs):
        model.train()

        for step, batch in enumerate(train_loader, start=1):
            batch = to_device(batch, cfg.device)

            logits, labels = model(
                batch["history"],
                batch["positions"],
                batch["mask"],
                batch["target"],
                cfg.num_negatives,
            )

            loss = F.cross_entropy(logits, labels)

            opt.zero_grad()
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                cfg.grad_clip,
            )
            opt.step()

            if step % cfg.log_every_n_steps == 0:
                print(
                    f"sampled softmax | "
                    f"epoch={epoch + 1}/{cfg.num_epochs} | "
                    f"step={step}/{len(train_loader)} | "
                    f"loss={loss.item():.6f} | "
                    f"grad_norm={float(grad_norm):.6f}"
                )

        metrics = validate_item_model(
            model,
            val_loader,
            cfg,
            sampled=True,
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": float(loss.item()),
            **metrics,
        })

        print(
            f"sampled softmax | "
            f"epoch={epoch + 1} done | "
            f"loss={loss.item():.6f} | "
            f"recall@{cfg.topk}={metrics[f'recall@{cfg.topk}']:.6f}"
        )

    return history

### Создание датасета

In [ ]:
cfg = Config(
    max_history_len=16,
    min_history_len=5,
    batch_size=1024,
    batch_size_val=64,
    d_model=128,
    d_ff=512,
    num_heads=4,
    num_layers=4,
    num_epochs=5,
    lr=1e-3,
    topk=10,
)

set_seed(cfg.seed)

train_examples, val_examples, item2idx, idx2item = prepare_examples(
    data=data,
    cfg=cfg,
)

val_examples = limit_examples(
    val_examples,
    cfg.val_subset_size,
    seed=cfg.seed,
)

num_items = len(item2idx)

train_loader = DataLoader(
    RecDataset(train_examples),
    batch_size=cfg.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    RecDataset(val_examples),
    batch_size=cfg.batch_size_val,
    shuffle=False,
    collate_fn=collate_fn,
)

print("device:", cfg.device)
print("num_items:", num_items)
print("train_examples:", len(train_examples))
print("val_examples:", len(val_examples))

### Обучение модели на полный softmax

In [ ]:
full_model = FullSoftmaxModel(
    num_items=num_items,
    cfg=cfg,
).to(cfg.device)

full_history = train_full_softmax(
    full_model,
    train_loader,
    val_loader,
    cfg,
)

full_history

### Обучение модели с равномерными негативами 

In [ ]:
sampled_model = SampledSoftmaxModel(
    num_items=num_items,
    cfg=cfg,
).to(cfg.device)

sampled_history = train_sampled_softmax(
    sampled_model,
    train_loader,
    val_loader,
    cfg,
)

sampled_history

## Модель на семантиках (2 балла)
- Перепишите пайплайн обучения и модель под работу с семантиками

---

В semantic-модели encoder кодирует историю пользователя:

$$
H = Encoder(h_1, h_2, \dots, h_T).
$$

Decoder генерирует Semantic ID авторегрессионно:

$$
p(s_i \mid h) =
\prod_{m=1}^{M}
p(s_{i,m} \mid s_{i,<m}, H).
$$

Во время обучения используется teacher forcing: на вход decoder-а подаются правильные предыдущие semantic-токены:

$$
[BOS, s_{i,1}, s_{i,2}, \dots, s_{i,M-1}].
$$

На позиции $m$ decoder должен предсказать токен $s_{i,m}$.

---

- Декодер имеет столько-же слоёв, сколько энкодер (в данной реализации 4)
- Каждый слой декодера состоит из Self Attention (как в энкодере) и Cross Attention, который смотрит на выход энкодра через KV проекцию

---

- Мы попробуем два варианта:
    1. обучать semantic-модель на исходных Semantic IDs, где возможны коллизии;
    2. построить `semantic_df_resolved`, где коллизии разрешены дополнительным токеном.
- Дополнительный токен делает Semantic ID уникальным, но он не обязательно несёт семантический смысл. Поэтому качество модели с таким токеном может стать хуже.

In [ ]:
def prepare_examples(data, semantic_df, cfg):
    sid_cols = get_sid_cols(semantic_df)

    # item_id -> internal index from 1 to num_items
    item2idx = ...
    idx2item = ...

    # join interactions with Semantic IDs and sort by user/time
    df = ...

    # last cfg.val_num_days are used for validation
    val_days = ...

    examples = []
    history = []
    current_uid = None

    for row in df.iter_rows(named=True):
        uid = ...
        item_id = ...
        day = ...

        # TODO: reset history when user changes

        # TODO: create example if history is long enough
        # example fields: history, target, sid, is_val

        # TODO: update user history

    train = ...
    val = ...

    return train, val, item2idx, idx2item


def collate_fn(batch):
    bsz = len(batch)
    max_len = ...

    # history: [B, L]
    # positions: [B, L]
    # mask: [B, L]
    history = ...
    positions = ...
    mask = ...

    # target: [B]
    # sid: [B, M]
    target = ...
    sid = ...

    for i, x in enumerate(batch):
        # TODO: fill padded tensors for one example
        ...

    return {
        "history": history,
        "positions": positions,
        "mask": mask,
        "target": target,
        "sid": sid,
    }

In [ ]:
cfg = Config(
    max_history_len=16,
    min_history_len=5,
    batch_size=...,
    batch_size_val=...,
    d_model=128,
    d_ff=512,
    num_heads=4,
    num_layers=4,
    num_epochs=...,
    lr=...,
    topk=10,
    num_levels=4,
    codebook_size=512,
    num_beams=100,
)

set_seed(cfg.seed)

train_examples, val_examples, item2idx, idx2item = prepare_examples(
    data=data,
    semantic_df=semantic_df_resolved,
    cfg=cfg,
)

val_examples = limit_examples(
    val_examples,
    cfg.val_subset_size,
    seed=cfg.seed,
)

num_items = len(item2idx)

train_loader = DataLoader(
    RecDataset(train_examples),
    batch_size=cfg.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4
)

val_loader = DataLoader(
    RecDataset(val_examples),
    batch_size=cfg.batch_size_val,
    shuffle=False,
    collate_fn=collate_fn,
)

print("device:", cfg.device)
print("num_items:", num_items)
print("train_examples:", len(train_examples))
print("val_examples:", len(val_examples))

In [ ]:
class SemanticTransformerModel(nn.Module):
    def __init__(self, num_items, cfg):
        super().__init__()

        self.num_levels = cfg.num_levels
        self.codebook_size = cfg.codebook_size
        self.bos_id = cfg.codebook_size

        self.encoder = HistoryTransformerEncoder(num_items, cfg)

        # TODO: embeddings for decoder input tokens
        # TODO: embeddings for semantic positions / levels
        # TODO: TransformerDecoder
        # TODO: prediction heads for each semantic level

    def make_decoder_input(self, sid):
        # sid: [B, M]
        # return: [B, M] = [BOS, sid_0, sid_1, ...]
        ...

    def causal_mask(self, T, device):
        # mask future decoder positions
        ...

    def forward(self, history, positions, mask, sid):
        # history: [B, L]
        # sid: [B, M]

        # TODO: encode history
        memory = ...

        # TODO: build decoder input tokens
        dec_tokens = ...

        # TODO: build decoder embeddings
        x = ...

        # TODO: run TransformerDecoder
        dec = ...

        losses = []
        preds = []

        for level in range(self.num_levels):
            # TODO: predict sid[:, level] from decoder output at this level
            logits = ...
            loss = ...

            losses.append(loss)
            preds.append(...)

        return {
            "loss": ...,
            "preds": ...,
        }

### Beam-Search (1 балл)
- На валидации нам нужно реализовать [beam-search](https://en.wikipedia.org/wiki/Beam_search), т.к. мы не можем прокинуть в декодер все возможные последовательности семантиков и посчитать их вероятности.

- Beam search хранит несколько наиболее вероятных частичных гипотез.

- `beam_size` — сколько гипотез мы храним во время поиска.
- `recall_k` — сколько финальных последовательностей используем для подсчёта recall.

Например, `beam_size=100` и `recall_k=10` означает: во время поиска мы держим до 100 гипотез, но метрику считаем только по top-10 финальным Semantic IDs.

In [ ]:
@torch.no_grad()
def semantic_beam_search(model, history, positions, mask, cfg, beam_size=10):
    model.eval()

    # memory: [B, L, D]
    memory = ...

    B = history.size(0)
    device = history.device

    # beams: [B, K, generated_len]
    beams = ...

    # scores: [B, K]
    scores = ...

    for level in range(model.num_levels):
        K = beams.size(1)

        # TODO: build decoder prefix from current beams
        # level=0: prefix contains only BOS
        # level>0: prefix contains BOS + generated tokens
        prev_tokens = ...

        # TODO: run decoder for all beams
        dec = ...

        # TODO: get log probabilities for the next semantic token
        log_probs = ...

        # TODO: extend beams and keep top beam_size hypotheses
        beams = ...
        scores = ...

    return beams, scores

In [ ]:
@torch.no_grad()
def validate_semantic_model(model, val_loader, cfg, beam_size=100, recall_k=10):
    model.eval()

    total = 0
    exact_hits = 0
    token_correct = 0
    token_total = 0
    top1_exact = 0

    for batch in val_loader:
        batch = to_device(batch, cfg.device)

        out = model(
            batch["history"],
            batch["positions"],
            batch["mask"],
            batch["sid"],
        )

        preds = out["preds"]

        token_correct += (preds == batch["sid"]).sum().item()
        token_total += batch["sid"].numel()
        top1_exact += (preds == batch["sid"]).all(1).sum().item()

        beams, scores = semantic_beam_search(
            model,
            batch["history"],
            batch["positions"],
            batch["mask"],
            cfg,
            beam_size=beam_size,
        )

        beams = beams[:, :recall_k, :]

        hit = (beams == batch["sid"][:, None, :]).all(-1).any(1)

        total += batch["sid"].size(0)
        exact_hits += hit.sum().item()

    return {
        "top1_sid_token_acc": token_correct / max(token_total, 1),
        "top1_sid_exact": top1_exact / max(total, 1),
        f"sid_recall_exact@{recall_k}": exact_hits / max(total, 1),
        "beam_size": beam_size,
    }

In [ ]:
def train_semantic(model, train_loader, val_loader, cfg, beam_size=100, recall_k=10):
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )

    history = []

    for epoch in range(cfg.num_epochs):
        model.train()

        for step, batch in enumerate(train_loader, start=1):
            batch = to_device(batch, cfg.device)

            out = model(
                batch["history"],
                batch["positions"],
                batch["mask"],
                batch["sid"],
            )

            loss = out["loss"]

            opt.zero_grad()
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                cfg.grad_clip,
            )
            opt.step()

            if step % cfg.log_every_n_steps == 0:
                preds = out["preds"]
                sid = batch["sid"]

                token_acc = (preds == sid).float().mean().item()
                exact_acc = (preds == sid).all(1).float().mean().item()

                print(
                    f"semantic | "
                    f"epoch={epoch + 1}/{cfg.num_epochs} | "
                    f"step={step}/{len(train_loader)} | "
                    f"loss={loss.item():.6f} | "
                    f"token_acc={token_acc:.6f} | "
                    f"exact_acc={exact_acc:.6f} | "
                    f"grad_norm={float(grad_norm):.6f}"
                )

        metrics = validate_semantic_model(
            model,
            val_loader,
            cfg,
            beam_size=beam_size,
            recall_k=recall_k,
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": float(loss.item()),
            **metrics,
        })

        metric_name = f"sid_recall_exact@{recall_k}"

        print(
            f"semantic | "
            f"epoch={epoch + 1} done | "
            f"loss={loss.item():.6f} | "
            f"{metric_name}={metrics[metric_name]:.6f} | "
            f"beam_size={beam_size}"
        )

    return history

## Обучите модель
- Модель на Semantic IDs должна превзойти **sampled softmax** baseline(не Full softmax) по `sid_recall_exact@10` / proxy-метрике, либо показать сопоставимое качество и объяснить результат.

- Важно: `sid_recall_exact@10` — это не полностью то же самое, что item-level `recall@10`, если в Semantic IDs есть коллизии.
- Обратите внимание, что модели на семантиках часто могут учиться дольше обычных, из-за архитектуры энкодера-декодера. Учитывайте это при обучении
- Подбирайте гиперпараметры

In [ ]:
semantic_model = SemanticTransformerModel(
    num_items=num_items,
    cfg=cfg,
).to(cfg.device)

semantic_history = train_semantic(
    semantic_model,
    train_loader,
    val_loader,
    cfg,
    beam_size=100,
    recall_k=10,
)

## Обучение с коллизиями (1 балл)
- Как уже писалось выше, коллизии не так страшны, как кажутся. Если модель сгенерила семантики, которые имеют несколько айтемов - можно взять их все, можно взять самый популярный или случайны item. На практике такой подход оказывается лучше, чем добавление дополнительного случайного семантика, который гораздо сложнее учить модели, т.к. он не имеет под собой семантического смысла.
- Обучите semantic-модель на исходном `semantic_df`, где полные Semantic IDs могут иметь коллизии.
- Не забудьте поставить num_levels=3.
- Сравните метрики всех 4 моделей и сделайте выводы
- *(Доп 2 балла) если сможете обогнать по качеству модель с полным softmax (Это может быть не так просто, лучше делать в самом конце)

In [ ]:
...

## RQ-VAE (2 балла)
- RQ-VAE — это более гибкий способ построения Semantic IDs. В отличие от RQ-KMeans, он обучает пространство, в котором происходит квантование.

---

Encoder переводит исходный embedding в latent-пространство:

$$
z_i = Encoder(x_i).
$$

Затем latent-вектор квантуется через residual quantization:

$$
z_{q,i} = c_{1, s_{i,1}} + c_{2, s_{i,2}} + \dots + c_{M, s_{i,M}}.
$$

Decoder восстанавливает исходный embedding:

$$
\hat{x}_i = Decoder(z_{q,i}).
$$

Базовый loss:

$$
\mathcal{L}
=
\|x_i - \hat{x}_i\|_2^2
+
\|sg[z_i] - z_{q,i}\|_2^2
+
\beta \|z_i - sg[z_{q,i}]\|_2^2,
$$

где $sg[\cdot]$ означает stop-gradient.

Первое слагаемое отвечает за качество восстановления embedding-а, второе обучает codebook-и, третье заставляет encoder не уходить слишком далеко от выбранных codebook-векторов.

---

- В RQ-VAE можно добавлять дополнительные лосы в модель, например лосс с коллаборативным сигналом. Есть статьи, показывающие важность коллаборативного сигнала в семантиках. [тык](https://arxiv.org/abs/2405.07314)

- В этом задании вам предстоит реализовать обучение базового RQ-VAE

In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)

    def __len__(self):
        return self.X.size(0)

    def __getitem__(self, idx):
        return self.X[idx]

In [ ]:
# Hint: для stop-gradient используйте .detach()

class RQVAE(nn.Module):
    def __init__(self, dim, hidden_dim=128, latent_dim=128, num_levels=3, codebook_size=512):
        super().__init__()

        self.num_levels = num_levels
        self.codebook_size = codebook_size
        self.latent_dim = latent_dim

        # TODO: encoder: x -> z
        self.encoder = ...

        # TODO: decoder: z_q -> x_hat
        self.decoder = ...

        # TODO: residual quantization codebooks
        self.codebooks = ...

    def quantize(self, z):
        # z: [B, latent_dim]
        residual = ...
        z_q = ...
        codes_all = []

        for codebook in self.codebooks:
            # TODO: find nearest codebook vector for residual
            codes = ...
            q = ...

            # TODO: update quantized sum and residual
            z_q = ...
            residual = ...

            codes_all.append(codes)

        codes_all = ...

        # straight-through estimator
        z_q_st = ...

        return z_q, z_q_st, codes_all

    def forward(self, x):
        # TODO: encode, quantize, decode
        z = ...
        z_q, z_q_st, codes = ...
        x_hat = ...

        return {
            "x_hat": ...,  # [B, dim]
            "z": ...,      # [B, latent_dim]
            "z_q": ...,    # [B, latent_dim]
            "codes": ...,  # [B, num_levels]
        }

In [ ]:
def train_rqvae(
    X,
    hidden_dim,
    latent_dim,
    num_levels,
    codebook_size,
    batch_size,
    num_epochs,
    lr,
    beta,
):
    device="cuda" if torch.cuda.is_available() else "cpu"
    X = np.asarray(X, dtype=np.float32)
    dim = X.shape[1]

    loader = DataLoader(
        EmbeddingDataset(X),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    model = RQVAE(
        dim=dim,
        hidden_dim=hidden_dim,
        latent_dim=latent_dim,
        num_levels=num_levels,
        codebook_size=codebook_size,
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        model.train()

        total_loss = 0.0
        total_recon = 0.0
        total_codebook = 0.0
        total_commit = 0.0
        total_count = 0

        for x in loader:
            x = x.to(device)

            out = model(x)

            # TODO: reconstruction loss
            recon_loss = ...
            
            # TODO: codebook loss
            codebook_loss = ...
            
            # TODO: commitment loss
            commit_loss = ...
            
            loss = recon_loss + codebook_loss + beta * commit_loss

            opt.zero_grad()
            loss.backward()
            opt.step()

            bs = x.size(0)
            total_loss += loss.item() * bs
            total_recon += recon_loss.item() * bs
            total_codebook += codebook_loss.item() * bs
            total_commit += commit_loss.item() * bs
            total_count += bs

        print(
            f"epoch={epoch + 1}/{num_epochs} | "
            f"loss={total_loss / total_count:.6f} | "
            f"recon={total_recon / total_count:.6f} | "
            f"codebook={total_codebook / total_count:.6f} | "
            f"commit={total_commit / total_count:.6f}"
        )

    return model

In [ ]:
@torch.no_grad()
def get_rqvae_semantic_ids(model, X, batch_size=8192, device=None):
    if device is None:
        device = next(model.parameters()).device

    X = np.asarray(X, dtype=np.float32)

    loader = DataLoader(
        EmbeddingDataset(X),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    model.eval()
    all_codes = []

    for x in loader:
        x = x.to(device)
        out = model(x)
        all_codes.append(out["codes"].cpu())

    return torch.cat(all_codes, dim=0).numpy().astype(np.int32)

In [ ]:
def train_rqvae_on_items(
    items_df,
    hidden_dim,
    latent_dim,
    num_levels,
    codebook_size,
    batch_size,
    num_epochs,
    lr,
    beta,
):
    item_ids = items_df["item_id"].to_numpy()
    X = np.vstack(items_df["normalized_embed"].to_list()).astype(np.float32)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = train_rqvae(
        X,
        hidden_dim=hidden_dim,
        latent_dim=latent_dim,
        num_levels=num_levels,
        codebook_size=codebook_size,
        batch_size=batch_size,
        num_epochs=num_epochs,
        lr=lr,
        beta=beta,
    )

    codes = get_rqvae_semantic_ids(
        model,
        X,
        batch_size=batch_size,
        device=device,
    )

    semantic_df = pl.DataFrame({
        "item_id": item_ids,
        **{f"sid_{i}": codes[:, i] for i in range(num_levels)},
    })

    return model, semantic_df

### Советы
- Не используйте слишком большие batch size. При большом batch size уменьшается число optimizer steps за эпоху, и codebook-и хуже исследуют пространство эмбеддингов. Это может привести к codebook collapse: многие item-ы начинают использовать небольшое число кодов.
- Учите много эпох

In [ ]:
rqvae_model, rqvae_semantic_df = train_rqvae_on_items(
    embeddings,
    hidden_dim=128,
    latent_dim=128,
    num_levels=3,
    codebook_size=512,
    batch_size=...,
    num_epochs=...,
    lr=...,
    beta=...,
)

rqvae_semantic_df.head()

- Чтобы получить баллы, нужно выбить метрику **unique semantic ids** выше, чем у RQ-KMeans

In [ ]:
tests.check_semantic_ids(rqvae_semantic_df)

## Модель на RQ-VAE семантиках
- Обучите модель, используя семантики из RQ-VAE
- Добавьте результаты в таблицу
- *(Доп 2 балла) если получится побить модель с семантиками из RQ-KMeans. (Это может оказаться не так просто)

## Итоговое сравнение (1 балл)

- Постройте табличку с результатами всех методов
- Сделайте выводы